# K-means, K-NN и GMM: проверяем руками

Ноутбук к теме [`lesson.md`](lesson.md). Здесь мы не только вызываем `sklearn`, но и отдельно смотрим на шаги Lloyd, responsibilities GMM и зависимость K-NN от числа соседей.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(7)


## 1. K-means и алгоритм Lloyd

Целевая функция K-means — сумма квадратов расстояний до центроидов. Алгоритм чередует assignment (ближайший центр) и update (среднее назначенных точек).

In [2]:
X, y_true = make_blobs(
    n_samples=360,
    centers=[(-2.1, -1.2), (0.1, 2.0), (2.2, -0.7)],
    cluster_std=[0.55, 0.62, 0.58],
    random_state=7,
)

model = KMeans(n_clusters=3, n_init=20, random_state=7).fit(X)
print(f"inertia={model.inertia_:.1f}")
print("centers:\n", np.round(model.cluster_centers_, 2))


inertia=230.3
centers:
 [[-2.13 -1.2 ]
 [ 2.24 -0.75]
 [ 0.09  1.87]]


In [3]:
def lloyd_trace(X, initial_centers, n_steps=8):
    centers = initial_centers.copy()
    trace = []
    for _ in range(n_steps):
        labels = np.argmin(((X[:, None] - centers[None, :]) ** 2).sum(axis=2), axis=1)
        trace.append((labels.copy(), centers.copy()))
        new_centers = np.array([X[labels == k].mean(axis=0) for k in range(len(centers))])
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
    return trace

initial = X[[0, 80, 160]]
trace = lloyd_trace(X, initial)
for step, (labels, centers) in enumerate(trace):
    inertia = sum(((X[labels == k] - centers[k]) ** 2).sum() for k in range(3))
    print(f"step={step}: inertia={inertia:.1f}")


step=0: inertia=600.9
step=1: inertia=231.2
step=2: inertia=230.4
step=3: inertia=230.3


## 2. Как выбирать число кластеров

Inertia всегда уменьшается при увеличении числа кластеров, поэтому посмотрим на неё вместе с silhouette.

In [4]:
rows = []
for k in range(2, 9):
    fit = KMeans(n_clusters=k, n_init=20, random_state=7).fit(X)
    rows.append((k, fit.inertia_, silhouette_score(X, fit.labels_)))

print(" K   inertia   silhouette")
for k, inertia, sil in rows:
    print(f"{k:2d}  {inertia:8.1f}   {sil:.3f}")


 K   inertia   silhouette
 2     914.2   0.555
 3     230.3   0.706
 4     200.7   0.580
 5     171.7   0.470
 6     147.5   0.336
 7     127.1   0.349
 8     110.9   0.355


Если истинные метки доступны только для проверки, номер кластера нельзя сравнивать с номером класса: названия можно переставить. ARI учитывает это автоматически.

In [5]:
print("ARI:", adjusted_rand_score(y_true, model.labels_))
print("silhouette:", silhouette_score(X, model.labels_))


ARI: 0.9916549962793194
silhouette: 0.7059620821325198


## 3. K-NN и bias–variance

K-NN почти ничего не делает на `fit`, но стоимость поиска соседей возникает при каждом запросе. Подберём `k` внутри кросс-валидации, причём стандартизация находится в том же pipeline.

In [6]:
X_moon, y_moon = make_moons(n_samples=500, noise=0.25, random_state=10)
ks = [1, 3, 5, 15, 30, 80]
for k in ks:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    scores = cross_val_score(pipe, X_moon, y_moon, cv=5)
    print(f"k={k:2d}: accuracy={scores.mean():.3f} ± {scores.std():.3f}")


k= 1: accuracy=0.902 ± 0.017
k= 3: accuracy=0.928 ± 0.012
k= 5: accuracy=0.936 ± 0.017
k=15: accuracy=0.932 ± 0.023
k=30: accuracy=0.914 ± 0.008
k=80: accuracy=0.896 ± 0.016


## 4. GMM и EM

GMM возвращает вероятности скрытой компоненты. В E-step считаются responsibilities, а в M-step по ним пересчитываются веса, средние и ковариации.

In [7]:
means = np.array([[-2.1, 0.0], [0.7, 1.8], [2.0, -1.0]])
covs = np.array([
    [[0.65, 0.32], [0.32, 0.48]],
    [[0.55, -0.22], [-0.22, 0.75]],
    [[0.65, 0.18], [0.18, 0.38]],
])
X_gmm = np.vstack([
    rng.multivariate_normal(mean, cov, size=180)
    for mean, cov in zip(means, covs)
])

gmm = GaussianMixture(
    n_components=3, covariance_type="full", n_init=10,
    reg_covar=1e-6, random_state=5,
).fit(X_gmm)
responsibilities = gmm.predict_proba(X_gmm)
print("first responsibilities:")
print(np.round(responsibilities[:5], 3))
print("BIC:", round(gmm.bic(X_gmm), 1))
print("average log-likelihood:", round(gmm.score(X_gmm), 3))


first responsibilities:
[[0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]]
BIC: 3436.2
average log-likelihood: -3.083


In [8]:
bic_by_k = {}
for k in range(1, 7):
    fit = GaussianMixture(n_components=k, covariance_type="full", n_init=10, random_state=5)
    fit.fit(X_gmm)
    bic_by_k[k] = fit.bic(X_gmm)
print("BIC by K:", {k: round(v, 1) for k, v in bic_by_k.items()})
print("best K by BIC:", min(bic_by_k, key=bic_by_k.get))


BIC by K: {1: np.float64(4046.6), 2: np.float64(3596.9), 3: np.float64(3436.2), 4: np.float64(3467.9), 5: np.float64(3495.0), 6: np.float64(3526.2)}
best K by BIC: 3


## 5. Сравнение

- K-means: жёсткие метки и центроиды, минимум within-cluster SSE.
- K-NN: supervised-правило по соседям, `k` управляет гладкостью.
- GMM: вероятностная смесь, мягкие принадлежности и ковариации.

Перед применением к новым данным нужно отдельно проверить масштабирование, форму кластеров, устойчивость и стоимость inference.